# 02 The sample
Which events are in the data, and which of them have enough price history. 

1. The list of cabinets coded by hand (`data/crises.csv`) is checked against ParlGov.
2. Parliamentary elections since 1997 and how many have at least 270 trading days of index history before the vote.
3. Check on the Israeli index (no Sunday observations on Yahoo).

In [1]:
import sys
from pathlib import Path

roots = [Path.cwd(), Path("/files"), Path.home()]
code_dir = next(p for r in roots for p in [r, *r.glob("**/elections_project/code")] if (p / "helpers.py").exists())
sys.path.insert(0, str(code_dir))
print("Working in", code_dir)

import pandas as pd

from helpers import (EVENTS_FROM, MIN_HISTORY, TICKERS, candidate_cabinets, eligible_from, load_elections,
                     load_prices, raw_file, DATA)

prices = load_prices()
elections = load_elections()
cabinets = pd.read_csv(raw_file("parlgov_view_cabinet"), parse_dates=["start_date", "election_date"])
crises = pd.read_csv(DATA / "crises.csv", dtype={"is_crisis": str})

Working in /files/mywork/elections_project/code


## 1. Cabinets: the hand-coded file against ParlGov

In [2]:
cand = candidate_cabinets(cabinets)
print(f"Cabinets since {EVENTS_FROM} that started without an election, 23 countries: {len(cand)}")

assert crises["event_id"].is_unique and crises["parlgov_cabinet_id"].is_unique
assert set(cand["cabinet_id"]) == set(crises["parlgov_cabinet_id"]), "crises.csv and ParlGov candidates differ"
print("crises.csv has the same", len(crises), "cabinets")

coded = crises[crises["is_crisis"].notna()]
print(f"coded: {len(coded)} | not coded (no index data): {len(crises) - len(coded)}")
print(coded["is_crisis"].value_counts().to_string())

Cabinets since 1997-01-01 that started without an election, 23 countries: 149
crises.csv has the same 149 cabinets
coded: 126 | not coded (no index data): 23
is_crisis
1            80
0            44
ambiguous     2


## 2. Elections and price history

In [3]:
elections = elections.assign(eligible_from=elections["country_name_short"].map({c: eligible_from(prices, c) for c in TICKERS}))
elections["eligible"] = elections["election_date"] >= elections["eligible_from"]

summary = (elections.groupby("country_name_short")
           .agg(elections=("election_id", "count"), eligible=("eligible", "sum"))
           .assign(lost=lambda d: d["elections"] - d["eligible"]))
print(summary.to_string())
print(f"Total: {summary['elections'].sum()} elections | eligible (>= {MIN_HISTORY} trading days): {summary['eligible'].sum()}")
print("Last election in the data:", elections["election_date"].max().date())

                    elections  eligible  lost
country_name_short                           
AUS                         9         9     0
AUT                         7         7     0
BEL                         6         6     0
CAN                         9         9     0
CHE                         6         6     0
DEU                         7         7     0
DNK                         8         2     6
ESP                         8         8     0
EST                         7         3     4
FIN                         7         3     4
FRA                         6         6     0
GBR                         7         7     0
GRC                        11        11     0
IRL                         6         5     1
ISR                        11        11     0
ITA                         6         6     0
JPN                         8         8     0
LTU                         6         2     4
NLD                         8         8     0
NZL                         8     

## 3. Israel: Yahoo has no Sunday observations
TASE traded Sunday to Thursday until 28 Dec 2025 and Monday to Friday after that. A Monday return on Yahoo is therefore a two-session return (Thursday close to Monday close), which matters for day 0 and for the threshold of a strong reaction.

In [4]:
isr = prices[TICKERS["ISR"]].dropna()
weekday = pd.Series(isr.index.dayofweek, index=isr.index).map({0: "Mon", 1: "Tue", 2: "Wed", 3: "Thu", 4: "Fri", 5: "Sat", 6: "Sun"})
print("Observations per weekday up to 26 Dec 2025:")
print(weekday[:"2025-12-26"].value_counts().reindex(["Sun", "Mon", "Tue", "Wed", "Thu", "Fri", "Sat"], fill_value=0).to_string())
print("From 1 Jan 2026:")
print(weekday["2026-01-01":].value_counts().reindex(["Mon", "Tue", "Wed", "Thu", "Fri"], fill_value=0).to_string())

Observations per weekday up to 26 Dec 2025:
date
Sun       0
Mon    1534
Tue    1510
Wed    1515
Thu    1525
Fri       2
Sat       0
From 1 Jan 2026:
date
Mon    45
Tue    42
Wed    43
Thu    43
Fri    40
